In [7]:
"""
使用 single_factor_test.py 的回测框架测试所有拍卖因子的预测能力
"""

import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path

# 导入测试框架
# from single_factor_test import backtest # 保留您的原始导入

# 导入拍卖因子模块
from Factor_ods import Factor # 保留您的原始导入


def load_real_data(filepath='train.csv'):
    """
    加载真实的 Optiver 数据

    Args:
        filepath: 数据文件路径
        nrows: 读取的行数

    Returns:
        pd.DataFrame 或 None
    """
    if not os.path.exists(filepath):
        print(f"⚠️  未找到真实数据文件: {filepath}")
        return None

    print("\n" + "=" * 80)
    print(f"加载真实数据: {filepath}")
    print("=" * 80)

    try:
        df = pd.read_csv(filepath)
        print(f"✅ 加载完成: {df.shape[0]} 行 × {df.shape[1]} 列")

        # 排序
        if 'stock_id' in df.columns and 'seconds_in_bucket' in df.columns:
            df = df.sort_values(['stock_id', 'seconds_in_bucket']).reset_index(drop=True)

        # 填充缺失值
        if 'near_price' in df.columns:
            df['near_price'] = df.groupby('stock_id')['near_price'].ffill()
        if 'far_price' in df.columns:
            df['far_price'] = df.groupby('stock_id')['far_price'].ffill()

        return df

    except Exception as e:
        print(f"❌ 加载失败: {e}")
        return None


# --- ▼▼▼ 更改后的函数 ▼▼▼ ---

def generate_auction_features(df, output_path="train_with_features.parquet"):
    """
    使用 Factor_update.py 生成所有拍卖特征, 并保存为 Parquet 文件

    Args:
        df: 原始 DataFrame
        output_path: Parquet 文件的保存路径

    Returns:
        df: 添加了拍卖特征的 DataFrame
        new_features: 新增特征列表
    """
    print("\n" + "=" * 80)
    print("生成拍卖特征 (Factor_update.py)")
    print("=" * 80)

    original_cols = df.columns.tolist()

    # -----------------------------------
    # 1. 生成所有拍卖特征
    # -----------------------------------
    # 静态因子（80多个）
    df = Factor.auction_all_features(df, group_col='stock_id')

    # 动态趋势因子（70+个）
    df = Factor.auction_dynamic_trend(df, group_col='stock_id')

    # 静态价格扩展（4个）
    df = Factor.auction_static_price_extensions(df, group_col='stock_id')
    # 获取新增特征
    new_features = [col for col in df.columns if col not in original_cols]

    print(f"\n✅ 特征生成完成")
    print(f"   原始列数: {len(original_cols)}")
    print(f"   最终列数: {len(df.columns)}")
    print(f"   新增特征: {len(new_features)}")

    # -----------------------------------
    # 2. 【您的要求】保存为 Parquet 文件
    # -----------------------------------
    print("\n" + "-" * 80)
    print(f"正在保存特征到 Parquet 文件: {output_path}")
    try:
        # 使用 'pyarrow' 引擎和 'snappy' 压缩，这是最高效的标准组合
        df.to_parquet(output_path, engine='pyarrow', compression='snappy')
        print(f"✅ 成功保存到: {output_path}")
        # 打印文件大小以便参考
        file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
        print(f"   文件大小: {file_size_mb:.2f} MB")
        
    except ImportError:
        print("❌ 保存失败: 请先安装 'pyarrow' 库 (pip install pyarrow)")
    except Exception as e:
        print(f"❌ 保存 Parquet 文件时出错: {e}")
    print("-" * 80)
    # --- ▲▲▲ 更改结束 ▲▲▲ ---

    return df, new_features


# --- 新增一个执行入口，用于运行脚本 ---
if __name__ == "__main__":
    
    print("--- 开始执行因子生成与保存任务 ---")
    
    # 1. 加载数据
    # (如果您的 train.csv 不在此目录，请修改路径)
    raw_df = load_real_data(filepath='train.csv')
    
    if raw_df is not None:
        # 2. 生成特征并自动保存
        # 最终文件将保存为 'train_auction_features.parquet'
        features_df, new_feature_list = generate_auction_features(
            raw_df, 
            output_path="train_auction_features.parquet"
        )
        
        print(f"\n脚本执行完毕。")
        print(f"共 {len(new_feature_list)} 个新特征已生成并保存。")
        
    else:
        print("数据加载失败，脚本终止。")

--- 开始执行因子生成与保存任务 ---

加载真实数据: train.csv
✅ 加载完成: 5237980 行 × 17 列

生成拍卖特征 (Factor_update.py)
1/7: 生成基础静态特征 (L1, 价格, 匹配效率)...
2/7: 生成时间紧迫性因子...
3/7: 生成不平衡方向性因子...
4/7: 生成价格冲击和效率因子...
5/7: 生成微观结构因子...
6/7: 生成高级交叉特征...
7/7: 生成融合（L1+拍卖池）静态因子...
警告: 缺少 'auc_bid_size' 或 'auc_ask_size'，跳过 'auction_fusion_factors' 融合因子。
1/1: 价格因子1
生成拍卖价格压力特征...
生成拍卖不平衡强度特征...
生成参考价格质量特征...

完成！共新增 85 个静态因子，当前总列数: 102
[trend] assigning ema/diff/ratio (4 columns)
[trend] generated factor: wap_trend_ema_fast_10
[trend] generated factor: wap_trend_ema_slow_30
[trend] generated factor: wap_trend_diff_10_30
[trend] generated factor: wap_trend_ratio_10_30
[trend] assigning signal/hist (2 columns)
[trend] generated factor: wap_trend_signal_9
[trend] generated factor: wap_trend_hist_10_30_9
[trend] assigning rolling slopes (3 columns)
[trend] generated factor: wap_trend_slope_20
[trend] generated factor: near_price_trend_slope_20
[trend] generated factor: wap_trend_vs_near_slope_20
[trend] assigning velocity/accelerati

In [ ]:
from pathlib import Path
import sys
import polars as pl
import pandas as pd
import gc # 引入垃圾回收模块
import os 

PROJECT_ROOT = Path(os.getcwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from single_factor_test import backtest, DEFAULT_EXCLUDE_COLS

base = Path(f"{PROJECT_ROOT}")
files = ["train_auction_features.parquet",
]

# Polars 数值类型集合（筛选因子列用）
NUM_DTYPES = {
    pl.Float32, pl.Float64,
    pl.Int8, pl.Int16, pl.Int32, pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
}

outputs_dir = PROJECT_ROOT / "outputs" / "final_factor_backtest"
outputs_dir.mkdir(parents=True, exist_ok=True)


for name in files:
    path = base / name
    print(f"\n=== {name} ===")
    if not path.exists():
        print(f"[跳过] 文件不存在: {path}")
        continue

    # 1) 惰性扫描以获取 schema
    try:
        ldf = pl.scan_parquet(str(path))
        schema = ldf.collect_schema()
        cols_set = set(schema.keys())
    except Exception as e:
        print(f"[跳过] 无法读取文件 schema: {e}")
        continue

    required = {"date_id", "seconds_in_bucket", "target"}
    missing = sorted(required - cols_set)
    if missing:
        print(f"[跳过] 缺少必要列 {missing}，该表不符合 single_factor_test 回测要求。")
        continue

    # 2) 基于 schema 自动选取数值型因子列
    excludes = set(DEFAULT_EXCLUDE_COLS)
    factor_cols = [
        c for c, dt in schema.items()
        if (c not in excludes) and (dt in NUM_DTYPES)
    ]
    if not factor_cols:
        print("[跳过] 未发现可用的数值因子列（已按 DEFAULT_EXCLUDE_COLS 过滤）。")
        continue

    print(f"文件 {name} 中发现 {len(factor_cols)} 个因子。准备分批回测...")

    # *** 内存优化：分批 (chunk) 读取和回测 ***
    all_results_list = []  # 用于收集每个批次的回测结果 (Pandas DataFrame)
    
    CHUNK_SIZE = 20  # <--- 在这里定义批次大小
    num_chunks = (len(factor_cols) + CHUNK_SIZE - 1) // CHUNK_SIZE

    for chunk_idx, i in enumerate(range(0, len(factor_cols), CHUNK_SIZE)):
        # 3) 获取当前批次的因子列表
        factor_chunk = factor_cols[i : i + CHUNK_SIZE]
        
        print(f"  [批次 {chunk_idx + 1}/{num_chunks}] 正在处理 {len(factor_chunk)} 个因子 (从 {factor_chunk[0]} 开始)")
        
        # 4) 仅读取必要列 + 当前这 *一批* 因子列
        cols_to_read = list(required) + factor_chunk
        
        # 预先声明变量，以便 finally 块可以访问
        pl_df = None
        df = None 

        try:
            # 5) 用 Polars 加载数据
            pl_df = (
                pl.scan_parquet(str(path))
                .select(cols_to_read)
                .collect(engine="streaming")
            )

            # 6) 在 Polars (内存中) 计算日期范围
            dmin, dmax = (
                pl_df
            .select(
                pl.col("date_id").cast(pl.Int64).min().alias("dmin"),
                pl.col("date_id").cast(pl.Int64).max().alias("dmax"),
            )
            .row(0)
                 )
            dmin = int(dmin); dmax = int(dmax)

            # 7) 转换为 Pandas (假设 backtest 函数需要 Pandas)
            #    这是你原有的逻辑，我保留它
            df = pl_df.to_pandas()
            
            # 8) 立即释放 Polars DataFrame 内存
            del pl_df
            pl_df = None # 确保变量被清除
            gc.collect() 

            # 9) 调用回测 (传入 Pandas df 和 *一批* 因子)
            accepted, processed = backtest(
                    df=df,
                    factor_cols=factor_chunk,
                    threshold=0.03,
                    start_date=dmin,
                    end_date=dmax,
                    check_inf=True,
            )

            if accepted is not None and not accepted.empty:
                all_results_list.append(accepted)
            else:
                print(f"  [警告] 批次 {chunk_idx + 1} 未返回有效结果。")

        except Exception as e:
            print(f"  [失败] 批次 {chunk_idx + 1} ({factor_chunk[0]}, ...) 回测执行异常: {e}")
            continue
        finally:
            # 确保在循环中显式删除 df 和 pl_df (以防万一)
            if pl_df is not None:
                del pl_df
            if df is not None:
                del df
            gc.collect() # 强制垃圾回收

    # (批次循环结束)
    
    # 10) 合并所有批次的结果并保存
    if not all_results_list:
        print(f"[完成] 文件 {name} 没有产出任何回测结果。")
        continue

    try:
        final_result = pd.concat(all_results_list)
        
        print(f"\n{name} 全部因子回测完成 (共 {len(final_result)} 行结果)。")
        print("回测结果(前10行):")
        print(final_result.head(10).to_string(index=False))

        out_csv = outputs_dir / f"{path.stem}_backtest.csv"
        final_result.to_csv(out_csv, index=False)
        print(f"[保存] 结果已写入: {out_csv}")
    
    except Exception as e:
        print(f"[失败] 合并或保存结果时出错: {e}")

print("\n=== 全部处理完毕 ===")


=== train_auction_features.parquet ===
文件 train_auction_features.parquet 中发现 179 个因子。准备分批回测...
  [批次 1/9] 正在处理 20 个因子 (从 imbalance_size 开始)
开始回测 20 个因子，日期范围 [0, 480]...

正在计算basic统计指标...

回测完成。总共 20 个因子，其中 6 个因子IC绝对值 >= 0.03

--- IC 绝对值 > 0.03 的过审因子排名 (按绝对值IC降序) ---
                           factor        ic    rankic      icir  rankicir
0  l1_bid_ask_liquidity_imbalance -0.122177 -0.132120 -1.455614 -1.557490
1                             wap -0.047282 -0.036299 -0.309481 -0.291787
2          near_ref_deviation_pct  0.043759  0.038101  0.343175  0.316532
3              near_ref_deviation  0.043716  0.038102  0.342569  0.316537
4           ref_nearfar_deviation -0.037282 -0.035381 -0.283836 -0.295024
5               far_ref_deviation  0.031012  0.032211  0.237798  0.273606
  [失败] 批次 1 (imbalance_size, ...) 回测执行异常: 'tuple' object has no attribute 'empty'
  [批次 2/9] 正在处理 20 个因子 (从 far_ref_deviation_pct 开始)
开始回测 20 个因子，日期范围 [0, 480]...

正在计算basic统计指标...

回测完成。总共 20 个因子，其中 6 个因子IC绝对值 >= 